In [1]:
from threading import Thread

import rclpy
from rclpy.callback_groups import ReentrantCallbackGroup
from rclpy.node import Node
from commander_py import commander
from commander_py import commander_utils
from geometry_msgs.msg import Pose, Point, Quaternion
from ur_commander.srv import VisualizePoses

# Initialize rclpy, handling already initialized case
try:
    rclpy.init()
except RuntimeError:
    pass

node = Node("ex_pose_goal")
callback_group = ReentrantCallbackGroup()
commander = commander.Commander(
    node=node,
    callback_group=callback_group,
    move_group="ur_manipulator",
    end_effector_frame=["camera_color_optical_frame", "tcp_ee"],
)
executor = rclpy.executors.MultiThreadedExecutor(2)
executor.add_node(node)
executor_thread = Thread(target=executor.spin, daemon=True)
executor_thread.start()
node.create_rate(1.0).sleep()

[INFO] [1753354213.716936743] [ex_pose_goal]: UR Commander Node Initialized


In [2]:
def display_poses(poses: list[Pose], frmae_id: str = "base_link") -> None:

    client = node.create_client(VisualizePoses, "/commander_viz/pose_visualization")
    while not client.wait_for_service(timeout_sec=3.0):
        node.get_logger().info("Waiting for /commander_viz/pose_visualization service...")
    client.call_async(VisualizePoses.Request(poses=poses, frame_id=frmae_id))

In [3]:
target = Pose(
    position=Point(x=0.5, y=0.0, z=0.5), orientation=Quaternion(x=0.0, y=0.0, z=0.0, w=1.0)
)

success = display_poses([target], "base_link")
if success:
    node.get_logger().info("Pose visualization request sent successfully.")

In [4]:
target = commander_utils.rotate_pose_euler(target, euler_deg=(0, 0, 270))
target = commander_utils.rotate_pose_euler(target, euler_deg=(180, 0, 0))
target_1 = commander_utils.translate_pose(target, translation=(0.0, 0.4, 0.0))
success = display_poses([target, target_1], "base_link")
# success = display_poses([target], "base_link")

In [ ]:
joint_values = [0.0, -1.57, 1.57, -1.57, -1.57, 0.0]
goal = commander.set_joint_goal(joint_values=joint_values)

In [8]:
goal = commander.set_pose_goal(
    pose=target, frame_id="base_link", ee_link="tcp_ee"
)
print(goal)

moveit_msgs.msg.MotionPlanRequest(workspace_parameters=moveit_msgs.msg.WorkspaceParameters(header=std_msgs.msg.Header(stamp=builtin_interfaces.msg.Time(sec=0, nanosec=0), frame_id='base_link'), min_corner=geometry_msgs.msg.Vector3(x=-1.0, y=-1.0, z=-1.0), max_corner=geometry_msgs.msg.Vector3(x=1.0, y=1.0, z=1.0)), start_state=moveit_msgs.msg.RobotState(joint_state=sensor_msgs.msg.JointState(header=std_msgs.msg.Header(stamp=builtin_interfaces.msg.Time(sec=1753354303, nanosec=697876808), frame_id='base_link'), name=['shoulder_lift_joint', 'elbow_joint', 'wrist_1_joint', 'wrist_2_joint', 'wrist_3_joint', 'shoulder_pan_joint'], position=[-1.6867934666075648, 1.752876106892721, -1.636789461175436, -1.5707767645465296, 0.39994820952415466, 0.3999505639076233], velocity=[0.0, -0.0, 0.0, 0.0, 0.0, 0.0], effort=[-2.966005563735962, -5.345225811004639, -0.8889327049255371, -0.2130710482597351, 0.07965397834777832, 0.6644030809402466]), multi_dof_joint_state=sensor_msgs.msg.MultiDOFJointState(hea

In [9]:
traj = commander.plan(
    pose_goal=goal,
    # joint_goal=goal,
    planner_id="PTP",
    pipeline_id="pilz_industrial_motion_planner",
    acc_scale=0.2,
    vel_scale=0.2,
)

[INFO] [1753354361.909042208] [ex_pose_goal]: Planning successful


In [10]:
success = commander.execute_trajectory(traj, wait_until_executed=True)
if success:
    node.get_logger().info("Trajectory executed successfully.")

[INFO] [1753354379.240548159] [ex_pose_goal]: Sending goal to action server /execute_trajectory with trajectory
[INFO] [1753354379.241897751] [ex_pose_goal]: Goal accepted by action server /execute_trajectory
[INFO] [1753354383.096744916] [ex_pose_goal]: Execution completed
[INFO] [1753354383.101540830] [ex_pose_goal]: Trajectory executed successfully.


In [12]:
sequence_goals = [
    commander.set_pose_goal(pose=target, frame_id="base_link", ee_link="tcp_ee"),
    commander.set_pose_goal(pose=target_1, frame_id="base_link", ee_link="tcp_ee"),
]
traj_sequence = commander.plan_sequence(
    pose_goals=sequence_goals,
    pipeline_id="pilz_industrial_motion_planner",
    planner_ids=["PTP", "LIN"],
    acc_scale=0.2,
    vel_scale=0.2,
)

[INFO] [1753354395.026046034] [ex_pose_goal]: Planning sequence successful


In [13]:
success = commander.execute_trajectory(traj_sequence, wait_until_executed=True)

[INFO] [1753354408.520079368] [ex_pose_goal]: Sending goal to action server /execute_trajectory with trajectory
[INFO] [1753354408.521150245] [ex_pose_goal]: Goal accepted by action server /execute_trajectory
[INFO] [1753354412.777165140] [ex_pose_goal]: Execution completed
